# Multidimensional analysis of GitHub repositories

This lab adapts the Week 04 data warehouse exercise to the GitHub repository data from `data-lab`. Each row represents **one repository at the 2026-09-28 snapshot**, not a transaction or a history of star counts. The warehouse can compare cohorts, but it cannot show how a repository's stars changed over time.

Run all cells from either this notebook's folder or the repository root. Requires `pandas` and `ipykernel`.

## 1. Business understanding

The research question is where popular repositories may leave developer needs unmet. Useful warehouse questions are: Which languages and ownership groups have the most abandoned repositories? Does that pattern vary by creation year? How many stars and open issues are attached to those cohorts?

**Stakeholders:** researchers and developers evaluating maintenance risk. **Grain:** one repository at the snapshot. **Dimensions:** creation time, language, owner type, license group, maintenance status. **Measures:** repository count, stars, forks, open issues, size in KB, and topic count. Stars and forks are snapshot counts. Summing them describes a cohort's total, while median stars describes a typical repository.

## 2. Data dictionary

| Field | Role | Meaning |
| --- | --- | --- |
| `repo_id` | key | GitHub repository identifier |
| `created_at` | source time | Repository creation timestamp, used for year, quarter, and month |
| `language` | dimension | Primary programming language |
| `owner_type` | dimension | User or organization |
| `license_group` | dimension | Grouped license from the preprocessing lab |
| `maintenance_status` | dimension | `abandoned`, `maintained`, or `unlabeled` from the preprocessing lab |
| `stars`, `forks`, `open_issues`, `size_kb`, `topic_count` | measures | Counts at the snapshot |

The time hierarchy is **month → quarter → year**. It groups repositories by *creation* date. `maintenance_status` is a snapshot label, not a timeless property. The prior lab found that its abandoned share is strongly confounded with repository age, so comparisons should retain creation year.

## 3. Data understanding

In [1]:
from pathlib import Path
import pandas as pd

# Jupyter may start in the notebook folder or at the repository root.
data_dir = next((p for p in (Path('../data-lab'), Path('data-lab'))
                 if (p / 'github_repos_2019_2025.csv').exists()), None)
if data_dir is None:
    raise FileNotFoundError('Run from dm-multidimensions/ or the repository root')

raw = pd.read_csv(data_dir / 'github_repos_2019_2025.csv', parse_dates=['created_at'])
clean = pd.read_csv(data_dir / 'github_repos_clean.csv',
                    usecols=['repo_id', 'license_group', 'maintenance_status'], keep_default_na=False)
print('Raw:', raw.shape, 'Cleaned:', clean.shape)
print('Snapshot dates:', raw['snapshot_date'].unique())
raw.head()

Raw: (25173, 21) Cleaned: (18866, 3)
Snapshot dates: <StringArray>
['2026-09-28']
Length: 1, dtype: str


,repo_id,full_name,owner_type,language,created_at,pushed_at,updated_at,stars,forks,open_issues,...,license,topics,topic_count,description,has_homepage,has_wiki,has_pages,has_discussions,archived,snapshot_date
0,165990457,Jinnrry/getAwayBSG,User,Go,2019-01-16 06:56:14+00:00,2019-09-05T15:10:49Z,2026-09-28T12:56:39Z,1148,116,8,...,AGPL-3.0,NaN,0,逃离北上广,True,True,True,False,True,2026-09-28
1,165957180,pingcap/dm,Organization,Go,2019-01-16 02:13:45+00:00,2022-05-09T08:28:36Z,2026-09-24T09:51:12Z,454,189,127,...,Apache-2.0,hacktoberfest,1,Data Migration Platform,False,True,False,False,True,2026-09-28
2,166091373,assafmo/xioc,User,Go,2019-01-16 18:38:35+00:00,2020-04-19T17:42:40Z,2026-08-10T15:37:37Z,163,9,4,...,MIT,command-line;command-line-tool;data-mining;def...,14,"Extract indicators of compromise from text, in...",False,True,False,False,False,2026-09-28
3,165960557,naqvijafar91/cuteDB,User,Go,2019-01-16 02:37:59+00:00,2022-06-26T22:31:19Z,2026-02-01T11:38:01Z,150,26,5,...,NaN,database;embedded-database;go;golang;key-value...,8,A slick BTree on disk based key value store im...,False,True,False,False,False,2026-09-28
4,166055089,giongto35/gowog,User,Go,2019-01-16 14:36:47+00:00,2019-05-09T14:55:57Z,2026-08-15T13:30:25Z,126,30,0,...,MIT,game;game-2d;golang;online-game;phaser;webgame...,7,"Gowog, Golang based Web multiplayer Online Game",True,True,False,False,False,2026-09-28


In [2]:
print('Missing keys:', raw['repo_id'].isna().sum(), clean['repo_id'].isna().sum())
print('Duplicate keys:', raw['repo_id'].duplicated().sum(), clean['repo_id'].duplicated().sum())
print(raw[['stars', 'forks', 'open_issues', 'size_kb', 'topic_count']].describe().round(1))
print(clean['maintenance_status'].value_counts())

Missing keys: 0 0
Duplicate keys: 0 0
          stars    forks  open_issues     size_kb  topic_count
count   25173.0  25173.0      25173.0     25173.0      25173.0
mean      154.2     28.5          8.5     33915.8          2.6
std      1551.9    278.8         94.9    365510.9          4.0
min        10.0      0.0          0.0         0.0          0.0
25%        14.0      2.0          0.0       159.0          0.0
50%        23.0      5.0          1.0      1201.0          0.0
75%        56.0     13.0          5.0      9017.0          4.0
max    110855.0  25050.0       9433.0  27295925.0         20.0
maintenance_status
abandoned     8074
unlabeled     7465
maintained    3327
Name: count, dtype: int64


## 4. Data preparation

Use the cleaned cohort selected in the Week 03 lab, then recover its original measures and creation timestamps from the raw file. The join is one-to-one on `repo_id`; this also prevents the preprocessing feature columns from being mistaken for warehouse measures.

In [3]:
fields = ['repo_id', 'full_name', 'created_at', 'language', 'owner_type',
          'stars', 'forks', 'open_issues', 'size_kb', 'topic_count']
warehouse = clean.merge(raw[fields], on='repo_id', how='left', validate='one_to_one')
assert len(warehouse) == len(clean)
assert warehouse[fields].notna().all().all()
assert (warehouse[['stars', 'forks', 'open_issues', 'size_kb', 'topic_count']] >= 0).all().all()
warehouse['year'] = warehouse['created_at'].dt.year
warehouse['quarter'] = 'Q' + warehouse['created_at'].dt.quarter.astype(str)
warehouse['month'] = warehouse['created_at'].dt.month
warehouse.head()

,repo_id,license_group,maintenance_status,full_name,created_at,language,owner_type,stars,forks,open_issues,size_kb,topic_count,year,quarter,month
0,165957180,Apache-2.0,abandoned,pingcap/dm,2019-01-16 02:13:45+00:00,Go,Organization,454,189,127,23101,1,2019,Q1,1
1,166091373,MIT,abandoned,assafmo/xioc,2019-01-16 18:38:35+00:00,Go,User,163,9,4,66,14,2019,Q1,1
2,165960557,None,abandoned,naqvijafar91/cuteDB,2019-01-16 02:37:59+00:00,Go,User,150,26,5,727,8,2019,Q1,1
3,166055089,MIT,unlabeled,giongto35/gowog,2019-01-16 14:36:47+00:00,Go,User,126,30,0,7951,7,2019,Q1,1
4,166048649,MIT,abandoned,emvi/hide,2019-01-16 13:54:17+00:00,Go,Organization,73,6,1,294,4,2019,Q1,1


In [4]:
print('Warehouse rows:', len(warehouse))
print('Missing values by field:')
print(warehouse.isna().sum())
print('Creation years:', sorted(warehouse['year'].unique()))
print('Languages:', sorted(warehouse['language'].unique()))

Warehouse rows: 18866
Missing values by field:
repo_id               0
license_group         0
maintenance_status    0
full_name             0
created_at            0
language              0
owner_type            0
stars                 0
forks                 0
open_issues           0
size_kb               0
topic_count           0
year                  0
quarter               0
month                 0
dtype: int64
Creation years: [np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
Languages: ['Go', 'Java', 'JavaScript', 'Python', 'Rust', 'TypeScript']


## 5. Modeling

### Star schema

```text
DimTime ───────┐
DimLanguage ────┤
DimOwner ───────┼── FactRepository (one row per repo_id and snapshot)
DimLicense ─────┤     repo_count, stars, forks, open_issues, size_kb, topic_count
DimStatus ──────┘
```

A snowflake would split `DimTime` into separate month, quarter, and year tables; here the small time dimension is simpler to keep together. A fact constellation would require another independent fact, such as issue events or historical snapshots. This dataset contains neither, so creating one would imply data we do not have.

In [5]:
dimension_columns = {
    'time': ['year', 'quarter', 'month'],
    'language': ['language'],
    'owner': ['owner_type'],
    'license': ['license_group'],
    'status': ['maintenance_status'],
}
dimensions = {}
fact_repository = warehouse[['repo_id', 'full_name']].copy()
for name, columns in dimension_columns.items():
    key = f'{name}_key'
    dimension = warehouse[columns].drop_duplicates().sort_values(columns).reset_index(drop=True)
    dimension.insert(0, key, range(1, len(dimension) + 1))
    dimensions[name] = dimension
    fact_repository = fact_repository.merge(
        warehouse[['repo_id', *columns]].merge(dimension, on=columns, validate='many_to_one')[['repo_id', key]],
        on='repo_id', validate='one_to_one')

fact_repository['repo_count'] = 1
for measure in ['stars', 'forks', 'open_issues', 'size_kb', 'topic_count']:
    fact_repository[measure] = warehouse.set_index('repo_id').loc[fact_repository['repo_id'], measure].to_numpy()

assert len(fact_repository) == len(warehouse)
assert fact_repository['repo_id'].is_unique
print('Fact table:', fact_repository.shape)
print('Dimension rows:', {name: len(table) for name, table in dimensions.items()})
fact_repository.head()

Fact table: (18866, 13)
Dimension rows: {'time': 81, 'language': 6, 'owner': 2, 'license': 5, 'status': 3}


,repo_id,full_name,time_key,language_key,owner_key,license_key,status_key,repo_count,stars,forks,open_issues,size_kb,topic_count
0,165957180,pingcap/dm,1,1,1,1,1,1,454,189,127,23101,1
1,166091373,assafmo/xioc,1,1,2,3,1,1,163,9,4,66,14
2,165960557,naqvijafar91/cuteDB,1,1,2,4,1,1,150,26,5,727,8
3,166055089,giongto35/gowog,1,1,2,3,3,1,126,30,0,7951,7
4,166048649,emvi/hide,1,1,1,3,1,1,73,6,1,294,4


In [6]:
# Reconstruct readable dimensions from the star schema for OLAP queries.
analysis = fact_repository.copy()
for name, dimension in dimensions.items():
    analysis = analysis.merge(dimension, on=f'{name}_key', validate='many_to_one')
assert len(analysis) == len(fact_repository)
analysis.head()

,repo_id,full_name,time_key,language_key,owner_key,license_key,status_key,repo_count,stars,forks,open_issues,size_kb,topic_count,year,quarter,month,language,owner_type,license_group,maintenance_status
0,165957180,pingcap/dm,1,1,1,1,1,1,454,189,127,23101,1,2019,Q1,1,Go,Organization,Apache-2.0,abandoned
1,166091373,assafmo/xioc,1,1,2,3,1,1,163,9,4,66,14,2019,Q1,1,Go,User,MIT,abandoned
2,165960557,naqvijafar91/cuteDB,1,1,2,4,1,1,150,26,5,727,8,2019,Q1,1,Go,User,None,abandoned
3,166055089,giongto35/gowog,1,1,2,3,3,1,126,30,0,7951,7,2019,Q1,1,Go,User,MIT,unlabeled
4,166048649,emvi/hide,1,1,1,3,1,1,73,6,1,294,4,2019,Q1,1,Go,Organization,MIT,abandoned


### Data cubes and OLAP operations

`repo_count` is additive across all dimensions. Stars, forks, issues, size, and topics are additive across disjoint repository groups at this single snapshot. A median is holistic: medians of subgroups cannot be combined into the overall median.

In [7]:
# Language × owner type × creation year cube. Each cell counts repositories.
cube = pd.pivot_table(analysis, values='repo_count',
                      index=['language', 'owner_type'], columns='year',
                      aggfunc='sum', fill_value=0, margins=True)
assert cube.iloc[-1, -1] == len(analysis)
cube

year                     2019  2020  2021  2022  2023  2024  2025    All
language   owner_type                                                   
Go         Organization    99   115    92    64    66    67    51    554
           User           146   169   129   109   101    97    74    825
Java       Organization    69    75    60    43    40    18    14    319
           User           170   163   142   104    94    65    39    777
JavaScript Organization   170   193   134    89    67    40    38    731
           User           517   480   417   346   292   235   204   2491
Python     Organization   240   341   291   292   341   351   363   2219
           User           875  1071   912   939  1001   942   777   6517
Rust       Organization    36    46    51    46    82    56    59    376
           User            81   121   115   118   126   116   112    789
TypeScript Organization   124   157   169   182   192   134   157   1115
           User           236   317   308   319   341   335   297   2153
All                      2763  3248  2820  2651  2743  2456  2185  18866

In [8]:
# Roll up from month to quarter, then drill down to months within quarters.
rollup = analysis.groupby(['year', 'quarter'], observed=True)['repo_count'].sum()
drilldown = analysis.groupby(['year', 'quarter', 'month'], observed=True)['repo_count'].sum()
assert rollup.equals(drilldown.groupby(level=['year', 'quarter']).sum())
print('Quarterly creation cohorts:')
display(rollup.to_frame('repositories'))
print('Monthly creation cohorts:')
display(drilldown.to_frame('repositories').head(12))

Quarterly creation cohorts:


repositories
year quarter              
2019 Q1                723
     Q2                756
     Q3                665
     Q4                619
2020 Q1                745
     Q2                941
     Q3                847
     Q4                715
2021 Q1                810
     Q2                811
     Q3                603
     Q4                596
2022 Q1                610
     Q2                752
     Q3                671
     Q4                618
2023 Q1                699
     Q2                791
     Q3                630
     Q4                623
2024 Q1                650
     Q2                646
     Q3                517
     Q4                643
2025 Q1                661
     Q2                836
     Q3                688

Monthly creation cohorts:


repositories
year quarter month              
2019 Q1      1               245
             2               220
             3               258
     Q2      4               245
             5               193
             6               318
     Q3      7               234
             8               201
             9               230
     Q4      10              180
             11              262
             12              177

In [9]:
# Slice: Python only. Dice: Python or TypeScript, user-owned, 2022–2024.
python_slice = analysis.loc[analysis['language'].eq('Python')]
dice = analysis.loc[
    analysis['language'].isin(['Python', 'TypeScript'])
    & analysis['owner_type'].eq('User')
    & analysis['year'].between(2022, 2024)
]
print('Python repositories:', len(python_slice))
print('Dice repositories:', len(dice))
display(dice.groupby(['year', 'language'], observed=True)['repo_count'].sum().unstack(fill_value=0))

Python repositories: 8736
Dice repositories: 3877


language,Python,TypeScript
year,,
2022,939,319
2023,1001,341
2024,942,335


In [10]:
# Pivot to view maintenance status by language. Keep unlabeled repositories visible.
pivot = analysis.pivot_table(values='repo_count', index='language',
                             columns='maintenance_status', aggfunc='sum', fill_value=0)
pivot['total'] = pivot.sum(axis=1)
pivot

maintenance_status,abandoned,maintained,unlabeled,total
language,,,,
Go,512,409,458,1379
Java,429,216,451,1096
JavaScript,1576,429,1217,3222
Python,3836,1158,3742,8736
Rust,403,324,438,1165
TypeScript,1318,791,1159,3268


### Measure classes and storage choices

| Measure | Class | Why |
| --- | --- | --- |
| Repository count, total stars, total forks | Distributive | Group totals can be summed. |
| Average stars | Algebraic | Combine subgroup sum and count. |
| Median stars | Holistic | Subgroup medians do not determine the overall median. |

Bitmap indexes suit low-cardinality dimensions such as `owner_type` and `maintenance_status`. Join indexes can speed repeated fact-to-dimension joins. Column storage helps scans of a few measures across many repositories. Those are database design options; this in-memory pandas lab does not create indexes.

| Architecture | Storage | Fit here |
| --- | --- | --- |
| ROLAP | Relational fact and dimension tables | Natural for the star schema above. |
| MOLAP | Precomputed multidimensional arrays | Fast repeated queries over fixed dimensions. |
| HOLAP | Relational detail plus precomputed aggregates | Useful when detail and common summaries are both needed. |

## 6. Evaluation

Compare counts, median stars, and total open issues by creation year and maintenance status. Year is included because the prior lab found a strong age effect in the status label. These aggregates describe this sampled corpus at one snapshot; they do not estimate a causal effect of language or ownership.

In [11]:
evaluation = analysis.groupby(['year', 'maintenance_status'], observed=True).agg(
    repositories=('repo_count', 'sum'),
    median_stars=('stars', 'median'),
    total_open_issues=('open_issues', 'sum'),
).reset_index()
evaluation['share_in_year_pct'] = (
    evaluation['repositories']
    / evaluation.groupby('year')['repositories'].transform('sum') * 100
).round(1)
assert evaluation['repositories'].sum() == len(analysis)
evaluation

,year,maintenance_status,repositories,median_stars,total_open_issues,share_in_year_pct
0,2019,abandoned,1702,28.0,19230,61.6
1,2019,maintained,333,64.0,9625,12.1
2,2019,unlabeled,728,20.0,2412,26.3
3,2020,abandoned,1800,27.0,15706,55.4
4,2020,maintained,449,51.0,13909,13.8
5,2020,unlabeled,999,19.0,4227,30.8
6,2021,abandoned,1285,29.0,9157,45.6
7,2021,maintained,455,41.0,21185,16.1
8,2021,unlabeled,1080,19.0,2390,38.3
9,2022,abandoned,1109,28.0,7370,41.8


## 7. More data cube computations

A full cube over language, owner type, and status includes every nonempty grouping set. The `ALL` value marks a rolled-up dimension. This is a small in-memory version of multiway aggregation.

In [12]:
from itertools import combinations

cube_dimensions = ['language', 'owner_type', 'maintenance_status']
full_cube_parts = []
for size in range(len(cube_dimensions) + 1):
    for group in combinations(cube_dimensions, size):
        if group:
            part = analysis.groupby(list(group), observed=True)['repo_count'].sum().reset_index()
        else:
            part = pd.DataFrame({'repo_count': [len(analysis)]})
        for column in cube_dimensions:
            if column not in group:
                part[column] = 'ALL'
        full_cube_parts.append(part[[*cube_dimensions, 'repo_count']])
full_cube = pd.concat(full_cube_parts, ignore_index=True)
assert full_cube.loc[(full_cube[cube_dimensions] == 'ALL').all(axis=1), 'repo_count'].item() == len(analysis)
print('Nonempty cube cells:', len(full_cube))
full_cube.head(12)

Nonempty cube cells: 84


,language,owner_type,maintenance_status,repo_count
0,ALL,ALL,ALL,18866
1,Go,ALL,ALL,1379
2,Java,ALL,ALL,1096
3,JavaScript,ALL,ALL,3222
4,Python,ALL,ALL,8736
5,Rust,ALL,ALL,1165
6,TypeScript,ALL,ALL,3268
7,ALL,Organization,ALL,5314
8,ALL,User,ALL,13552
9,ALL,ALL,abandoned,8074


## 8. Iceberg cube

Keep only language × owner × status cells with at least 100 repositories. Filtering after aggregation illustrates the iceberg result. BUC computes qualifying cells more efficiently on high-dimensional data by pruning branches; this small lab uses pandas groupby rather than implementing BUC.

In [13]:
threshold = 100
iceberg = full_cube.loc[
    full_cube[cube_dimensions].ne('ALL').all(axis=1)
    & full_cube['repo_count'].ge(threshold)
].sort_values('repo_count', ascending=False).reset_index(drop=True)
print(f'Cells with at least {threshold} repositories:', len(iceberg))
iceberg

Cells with at least 100 repositories: 35


,language,owner_type,maintenance_status,repo_count
0,Python,User,unlabeled,2971
1,Python,User,abandoned,2932
2,JavaScript,User,abandoned,1175
3,JavaScript,User,unlabeled,1048
4,Python,Organization,abandoned,904
5,TypeScript,User,unlabeled,883
6,TypeScript,User,abandoned,879
7,Python,Organization,unlabeled,771
8,Python,User,maintained,614
9,Python,Organization,maintained,544


## 9. Conclusion

The warehouse supports cohort comparisons by creation time, language, ownership, license, and maintenance status. It keeps the repository grain and snapshot date explicit. The earlier preprocessing lab found that status is age-confounded and star counts are heavily skewed, so year-stratified counts and median stars are safer summaries than an overall mean. A true trend analysis of stars or maintenance would require multiple snapshots.

## 10. Assignment checks

1. Which dimensions identify a creation cohort, and which measures describe its size and activity?
2. Compare the abandoned share across creation years in the evaluation table. Why does the label rule affect that pattern?
3. Use the cube to compare Python user-owned and organization-owned repositories by year.
4. Change the iceberg threshold and explain which cells disappear.
5. Explain why the current dataset cannot support a fact constellation or a history of stars.